### Loading Datasets

In [2]:
import pandas as pd
import numpy as np

# 1. Load the datasets and convert timestamps to datetime objects
telemetry_df = pd.read_csv('sdwan_telemetry.csv', parse_dates=['timestamp'])
labels_df = pd.read_csv('ml_ground_truth_labels.csv', parse_dates=['timestamp'])

# 2. Sort by time (Crucial for time-series ML)
telemetry_df = telemetry_df.sort_values('timestamp').reset_index(drop=True)
labels_df = labels_df.sort_values('timestamp').reset_index(drop=True)

# 3. Verify the ingestion
print(f"✅ Telemetry Records Loaded: {len(telemetry_df)}")
print(f"✅ Fault Events Loaded: {len(labels_df)}")

# Display the first 5 rows of our network data
display(telemetry_df.head())

✅ Telemetry Records Loaded: 86
✅ Fault Events Loaded: 30


,timestamp,avg_latency_ms,jitter_ms,status
0,2026-06-25 21:14:03,0.164,0.0,UP
1,2026-06-25 21:14:08,0.148,0.0,UP
2,2026-06-25 21:14:13,0.136,0.0,UP
3,2026-06-25 21:14:19,0.146,0.0,UP
4,2026-06-25 21:14:24,0.149,0.0,UP


,timestamp,fault_type,severity,target_node
0,2026-06-25 21:14:27,packet_loss,50_percent,p-core
1,2026-06-25 21:14:47,fault_cleared,0,p-core
2,2026-06-25 21:15:08,packet_loss,50_percent,p-core
3,2026-06-25 21:15:28,fault_cleared,0,p-core
4,2026-06-25 21:15:48,packet_loss,50_percent,p-core


### Time-Series Merging & Precursor Engineering

In [3]:
# 1. Merge labels into telemetry using a backward time-series search
# This tells every telemetry ping what the "active" network state was at that exact second.
merged_df = pd.merge_asof(telemetry_df, labels_df, on='timestamp', direction='backward')

# 2. Fill initial states (Assume healthy before the first injected fault)
merged_df['fault_type'] = merged_df['fault_type'].fillna('fault_cleared')

# 3. Create the master Target Label column
merged_df['ml_label'] = 'Healthy'
merged_df.loc[merged_df['fault_type'] == 'packet_loss', 'ml_label'] = 'Failing'

# 4. THE MAGIC: Create the "Precursor" State
# Find the exact moments where the network transitions from Healthy to Failing
failure_starts = (merged_df['ml_label'] == 'Failing') & (merged_df['ml_label'].shift(1) != 'Failing')

# Look back 3 telemetry cycles (~15 seconds) BEFORE the crash and label them as "Precursor"
for i in merged_df[failure_starts].index:
    start_idx = max(0, i - 3)
    # Ensure we only overwrite 'Healthy' states with 'Precursor', not actual failures
    merged_df.loc[start_idx:i-1, 'ml_label'] = 'Precursor'

# 5. Clean up redundant columns
final_df = merged_df[['timestamp', 'avg_latency_ms', 'jitter_ms', 'ml_label']].copy()

# 6. Print the distribution of our AI training labels
print("📊 AI Training Data Distribution:")
print(final_df['ml_label'].value_counts())

# Display a sneak peek of a transition period!
transition_view = final_df[(final_df['ml_label'] == 'Precursor') | (final_df['ml_label'] == 'Failing')].head(8)
display(transition_view)

📊 AI Training Data Distribution:
ml_label
Precursor    45
Failing      29
Healthy      12
Name: count, dtype: int64


,timestamp,avg_latency_ms,jitter_ms,ml_label
2,2026-06-25 21:14:13,0.136,0.0,Precursor
3,2026-06-25 21:14:19,0.146,0.0,Precursor
4,2026-06-25 21:14:24,0.149,0.0,Precursor
5,2026-06-25 21:14:29,999.000,999.0,Failing
6,2026-06-25 21:14:44,0.102,0.0,Failing
8,2026-06-25 21:14:55,0.166,0.0,Precursor
9,2026-06-25 21:15:00,0.159,0.0,Precursor
10,2026-06-25 21:15:05,0.155,0.0,Precursor


### Train the Predictive Engine

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

# 1. Prepare Features (X) and Target (y)
# Drop the timestamp (not a feature) and ml_label (our target)
X = final_df[['avg_latency_ms', 'jitter_ms']]

# 2. Translate text labels into ML-readable numbers
# 0 = Healthy, 1 = Precursor (About to fail), 2 = Failing
label_map = {'Healthy': 0, 'Precursor': 1, 'Failing': 2}
y = final_df['ml_label'].map(label_map)

# 3. Split the Data
# Give the model 80% of the data to learn from, and hide 20% to test it later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"📚 Training the model on {len(X_train)} network states...")

# 4. Initialize and Train the Random Forest
# We use 'balanced' class weights because Precursor states are rare compared to Healthy states
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
print("✅ AI Model Successfully Trained!\n")

📚 Training the model on 68 network states...
✅ AI Model Successfully Trained!

📊 AI Predictive Performance Report:
               precision    recall  f1-score   support

  Healthy (0)       0.33      1.00      0.50         3
Precursor (1)       0.80      0.44      0.57         9
  Failing (2)       1.00      0.67      0.80         6

     accuracy                           0.61        18
    macro avg       0.71      0.70      0.62        18
 weighted avg       0.79      0.61      0.64        18



### Testing the Model

In [ ]:
# 5. The Final Exam: Test the model on the hidden 20%
predictions = rf_model.predict(X_test)

# 6. Print the Report Card
print("📊 AI Predictive Performance Report:")
target_names = ['Healthy (0)', 'Precursor (1)', 'Failing (2)']
print(classification_report(y_test, predictions, target_names=target_names))

### Final Validation & Serialization

In [6]:
import joblib
from sklearn.metrics import confusion_matrix

print("🔍 PHASE 3: FINAL EVALUATION METRICS")
print("-" * 40)

# 1. Calculate False Positive Rate (FPR) for the Precursor state
cm = confusion_matrix(y_test, predictions)
# For Precursor (Index 1): 
# FP = Predicted Precursor, but actually Healthy (Index 0,1)
# TN = Actual Healthy, Predicted Healthy (Index 0,0)
false_positives = cm[0][1] 
true_negatives = cm[0][0]

if (false_positives + true_negatives) > 0:
    fpr = false_positives / (false_positives + true_negatives)
else:
    fpr = 0.0

print(f"⚠️ False Positive Rate (Precursor): {fpr:.2%} (Lower is better)")

# 2. Calculate Prediction Lead Time
# We engineered the precursor to trigger 3 telemetry cycles before failure.
# Telemetry runs every 5 seconds.
lead_time = 3 * 5
print(f"⏱️ Average Prediction Lead Time:  {lead_time} seconds before crash")
print("-" * 40)

# 3. Serialize (Save) the Model
model_filename = 'predictive_brain.joblib'
joblib.dump(rf_model, model_filename)
print(f"💾 SUCCESS: Model frozen and saved to disk as '{model_filename}'")

🔍 PHASE 3: FINAL EVALUATION METRICS
----------------------------------------
⚠️ False Positive Rate (Precursor): 0.00% (Lower is better)
⏱️ Average Prediction Lead Time:  15 seconds before crash
----------------------------------------
💾 SUCCESS: Model frozen and saved to disk as 'predictive_brain.joblib'
